# AIR Phase 3 Notebook

Use this notebook for your own run. The setup cells, file checks, report readers, and packaging commands are already in place. Fill the TODO blocks where the retrieval representation, hybrid merge, reranking text, or fusion logic is part of the project work.


# Stage 0

The prepared ABO-Home catalog is included in the project folder. Run the check below before using the catalog in the later stages.


In [2]:
# Extract the fixed image subset once, then run the catalog check.
from pathlib import Path
import zipfile

image_zip = Path("data/AIR_Phase3_stage1_images.zip")
image_root = Path("data/raw/images/small")
if image_zip.exists() and not any(image_root.rglob("*.jpg")):
    print("extracting", image_zip)
    with zipfile.ZipFile(image_zip) as zf:
        zf.extractall(".")

!python src/tests.py --stage 0 --catalog data/catalog_subset.parquet


python3: can't open file '/content/src/tests.py': [Errno 2] No such file or directory


# Stage 1


# Stage 1 — Single-Vector Multimodal Dense + SPLADE++ Sparse Embeddings & Indexes

This notebook builds the first-stage search representation for the ABO-Home-2K
catalog produced in Stage 0.

For each product it creates:

1. **one single-vector multimodal dense embedding** from the combined
   `{"text": product_text, "image": image_path}` input (no manual text/image
   averaging — the model fuses modalities internally);
2. **one SPLADE++-style sparse embedding** from `product_text`;
3. **searchable dense (FAISS) and sparse (SciPy CSR) indexes**.

### Required outputs
```
artifacts/embeddings/product_dense.npy
artifacts/embeddings/product_sparse.jsonl
artifacts/embeddings/product_ids.txt
artifacts/index/dense_index.faiss
artifacts/index/sparse_index.npz
artifacts/index/index_manifest.json
reports/indexing_summary.md
```

`product_ids.txt` is the shared **product-ID mapping**: FAISS row `i` ↔ CSR row
`i` ↔ `product_ids.txt` line `i` ↔ `product_dense.npy` row `i`.

### How to run on Colab
1. Put your Stage 1 inputs in a Google Drive folder named
   **`AIR_Phase3_stage1_input`** containing:
   - `catalog_subset.parquet`
   - the product images as a **`.zip`** (e.g. `raw.zip` of `data/raw`) or a folder tree.
2. Use a **GPU runtime** (the dense VLM needs ~8 GB VRAM; a T4 is fine with
   the batch size you select for the run).
3. Run top to bottom. The notebook mounts Drive, copies the inputs to local
   disk (`/content`), unzips the images there for fast reads, then writes all
   artifacts under `artifacts/` and `reports/`.


## 0. Install dependencies

In [ ]:
# Install the retrieval and indexing runtime from pyproject.toml.
import os
import subprocess
import sys

try:
    import tomllib
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tomli"])
    import tomli as tomllib

PROJECT_FILE = "pyproject.toml"
with open(PROJECT_FILE, "rb") as f:
    project_cfg = tomllib.load(f)

deps = list(project_cfg["project"].get("dependencies", []))
deps += list(project_cfg["project"].get("optional-dependencies", {}).get("stage1-gpu", []))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *deps])
print("dependencies installed from", PROJECT_FILE)


## 1. Mount Drive, stage inputs onto local disk & resolve images


here is the link to the raw data (output of stage 0): 
https://drive.google.com/drive/folders/1Z_kG20b3snyV8nEszNdpDiq4wxmKlL0C?usp=sharing

In [ ]:
import os, json, time, glob, shutil, zipfile
import numpy as np
import pandas as pd

# ----- mount Google Drive -----
from google.colab import drive
drive.mount("/content/drive")

# TODO: set the Drive folder that contains the Stage 1 catalog and images.
DRIVE_INPUT = "..."
WORK        = "/content/stage1_input"
os.makedirs(WORK, exist_ok=True)

# ----- output locations -----
EMB_DIR    = "artifacts/embeddings"
INDEX_DIR  = "artifacts/index"
REPORT_DIR = "reports"
for d in (EMB_DIR, INDEX_DIR, REPORT_DIR):
    os.makedirs(d, exist_ok=True)

DENSE_NPY    = f"{EMB_DIR}/product_dense.npy"
SPARSE_JSONL = f"{EMB_DIR}/product_sparse.jsonl"
IDS_TXT      = f"{EMB_DIR}/product_ids.txt"
DENSE_INDEX  = f"{INDEX_DIR}/dense_index.faiss"
SPARSE_INDEX = f"{INDEX_DIR}/sparse_index.npz"
MANIFEST     = f"{INDEX_DIR}/index_manifest.json"
SUMMARY      = f"{REPORT_DIR}/indexing_summary.md"

# Dense and sparse encoders used for this run.
DENSE_MODEL  = "nvidia/llama-nemotron-embed-vl-1b-v2"
SPARSE_MODEL = "prithivida/Splade_PP_en_v1"

# Run knobs. Lower DENSE_BATCH_SIZE/DENSE_CHUNK on smaller GPUs.
DENSE_BATCH_SIZE  = 2
DENSE_CHUNK       = 32
SPARSE_BATCH_SIZE = 16
SPARSE_MAX_LEN    = 256
SPARSE_TOP_N      = 128
DENSE_NORMALIZE   = True

print("Drive input:", DRIVE_INPUT)
print("dense:", DENSE_MODEL, "| sparse:", SPARSE_MODEL)


In [ ]:
# TODO: verify the catalog and image tree resolve correctly for your input folder.
# ----- unzip any archives in the Drive folder (e.g. raw.zip) to local disk --
os.makedirs(WORK, exist_ok=True)
zips = sorted(glob.glob(os.path.join(DRIVE_INPUT, "*.zip")))
for z in zips:
    local_zip = os.path.join(WORK, os.path.basename(z))
    print("copying zip:", z)
    shutil.copyfile(z, local_zip)
    print("unzipping", os.path.basename(z), "...")
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(WORK)
if not zips:
    print("no .zip in Drive folder; will read images directly from Drive (slower)")

def _find(name, *roots):
    """Find a file by name directly under, or anywhere within, the given roots."""
    for root in roots:
        direct = os.path.join(root, name)
        if os.path.exists(direct):
            return direct
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if hits:
            return sorted(hits)[0]
    return None

# ----- locate catalog_subset.parquet (Drive folder, then unzipped tree) ----
src_parquet = _find("catalog_subset.parquet", DRIVE_INPUT, WORK)
assert src_parquet, ("catalog_subset.parquet not found in " + DRIVE_INPUT +
                     " or inside any uploaded zip.")
CATALOG_PATH = os.path.join(WORK, "catalog_subset.parquet")
if os.path.abspath(src_parquet) != os.path.abspath(CATALOG_PATH):
    shutil.copyfile(src_parquet, CATALOG_PATH)
print("catalog:", src_parquet)

# ----- build a basename -> path index over the images (layout-agnostic) ----
# Works whether raw.zip expands to data/raw/images/small/... or raw/images/...
IMAGES_BASE = WORK if zips else DRIVE_INPUT
print("indexing image files under", IMAGES_BASE, "...")
IMAGE_INDEX = {}
for root, _, files in os.walk(IMAGES_BASE):
    for fn in files:
        if fn.lower().endswith((".jpg", ".jpeg", ".png")):
            IMAGE_INDEX.setdefault(fn, os.path.join(root, fn))
print("images indexed:", len(IMAGE_INDEX))

def resolve_image(path):
    cand = os.path.join(IMAGES_BASE, path)        # fast path if tree is already present
    if os.path.exists(cand):
        return cand
    if os.path.isabs(path) and os.path.exists(path):
        return path
    return IMAGE_INDEX.get(os.path.basename(path))   # robust: match by filename

# ----- load catalog & verify required columns -------------
catalog = pd.read_parquet(CATALOG_PATH)
REQUIRED = ["product_id", "title", "product_text", "image_path"]
missing = [c for c in REQUIRED if c not in catalog.columns]
assert not missing, f"catalog missing required columns: {missing}"
print(f"catalog rows: {len(catalog)}")

catalog["resolved_image"] = catalog["image_path"].map(resolve_image)
n_missing_img = int(catalog["resolved_image"].isna().sum())
print(f"images resolved: {len(catalog) - n_missing_img}/{len(catalog)} "
      f"(missing: {n_missing_img})")
assert n_missing_img == 0, "Some images not found - check raw.zip contents."

# Stable ordering: product_ids.txt line i <-> product_dense.npy row i
product_ids   = catalog["product_id"].tolist()
product_texts = catalog["product_text"].tolist()
image_paths   = catalog["resolved_image"].tolist()

## 2. Dense multimodal embeddings

The dense vector is produced **directly** from the combined
`{"text": product_text, "image": image_path}` object via `encode_document`
(falling back to `encode`). We never encode text and image separately and merge
them. Embeddings are **checkpointed every `DENSE_CHUNK` products** so a Colab
disconnect can resume instead of restarting, and the final `.npy` is cached so
later retrieval runs never recompute it.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("device:", DEVICE, "|", GPU_NAME)

try:
    dense_model = SentenceTransformer(DENSE_MODEL, trust_remote_code=True,
                                       device=DEVICE, model_kwargs={"torch_dtype": torch.float16}
                                       if DEVICE == "cuda" else {})
    DENSE_LOAD_CFG = {"trust_remote_code": True, "device": DEVICE, "dtype": "float16" if DEVICE == "cuda" else "default"}
except Exception as e:
    print("float16 load failed, falling back to default dtype:", e)
    dense_model = SentenceTransformer(DENSE_MODEL, trust_remote_code=True, device=DEVICE)
    DENSE_LOAD_CFG = {"trust_remote_code": True, "device": DEVICE, "dtype": "default"}

print("loaded:", DENSE_MODEL)


In [ ]:
def l2_normalize(mat):
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms

def encode_documents(inputs, batch_size):
    """Encode product dictionaries into one dense vector per product."""
    kw = dict(batch_size=batch_size, show_progress_bar=False,
              convert_to_numpy=True, normalize_embeddings=DENSE_NORMALIZE)
    fn = getattr(dense_model, "encode_document", None)
    encode_fn = fn if fn is not None else dense_model.encode
    vectors = encode_fn(inputs, **kw)
    return np.asarray(vectors, dtype=np.float32)

# Build the multimodal product inputs: one dict per product.
product_inputs = [{"text": t, "image": p}
                  for t, p in zip(product_texts, image_paths)]

# Smoke test on 2 products before the full run.
_smoke = encode_documents(product_inputs[:2], batch_size=2)
print("smoke embedding shape:", _smoke.shape)
DENSE_DIM = int(_smoke.shape[1])
print("dense dimension:", DENSE_DIM)


In [ ]:
from tqdm.auto import tqdm

def build_dense(inputs, out_npy, ids_txt, chunk, batch_size):
    # Cache: never recompute embeddings on later runs.
    if os.path.exists(out_npy) and os.path.exists(ids_txt):
        print("dense cache hit -> loading", out_npy)
        return np.load(out_npy)

    part_npy, part_cnt = out_npy + ".part.npy", out_npy + ".part.cnt"
    parts, done = [], 0
    if os.path.exists(part_npy) and os.path.exists(part_cnt):
        parts.append(np.load(part_npy))
        done = int(open(part_cnt).read().strip())
        print(f"resuming dense from {done}/{len(inputs)}")

    for s in tqdm(range(done, len(inputs), chunk), desc="dense"):
        batch = inputs[s:s + chunk]
        vecs = encode_documents(batch, batch_size)
        parts.append(vecs)
        done = s + len(batch)
        np.save(part_npy, np.vstack(parts).astype(np.float32))
        with open(part_cnt, "w") as f:
            f.write(str(done))

    dense = np.vstack(parts).astype(np.float32)
    np.save(out_npy, dense)
    # IDs written in the exact catalog row order: row i <-> line i.
    with open(ids_txt, "w") as f:
        f.write("\n".join(product_ids) + "\n")
    for p in (part_npy, part_cnt):
        if os.path.exists(p):
            os.remove(p)
    return dense

t0 = time.time()
product_dense = build_dense(product_inputs, DENSE_NPY, IDS_TXT,
                            DENSE_CHUNK, DENSE_BATCH_SIZE)
DENSE_MINUTES = (time.time() - t0) / 60
print("dense matrix:", product_dense.shape, "| minutes:", round(DENSE_MINUTES, 1))

assert product_dense.shape[0] == len(catalog), "one dense vector per product"
DENSE_DIM = int(product_dense.shape[1])
_norms = np.linalg.norm(product_dense, axis=1)
print("norm mean/min/max: %.4f / %.4f / %.4f" %
      (_norms.mean(), _norms.min(), _norms.max()))


## 3. Sparse SPLADE++ embeddings

SPLADE weight per vocab term = `max_t log(1 + relu(logits_t))` over the
sequence (attention-masked). We keep the **top-`SPARSE_TOP_N`** non-zero dims
per product and store them as `{product_id, indices, values}` — never a dense
30k array.

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

sp_tokenizer = AutoTokenizer.from_pretrained(SPARSE_MODEL)
sp_model = AutoModelForMaskedLM.from_pretrained(SPARSE_MODEL).to(DEVICE).eval()
VOCAB_SIZE = int(sp_model.config.vocab_size)
print("loaded sparse model:", SPARSE_MODEL, "| vocab:", VOCAB_SIZE)

@torch.no_grad()
def splade_encode_batch(texts, top_n):
    enc = sp_tokenizer(texts, padding=True, truncation=True,
                       max_length=SPARSE_MAX_LEN, return_tensors="pt").to(DEVICE)

    logits = sp_model(**enc).logits
    activations = torch.log1p(torch.relu(logits))
    mask = enc["attention_mask"].unsqueeze(-1)
    pooled, _ = (activations * mask).max(dim=1)
    pooled = pooled.cpu().numpy()

    results = []
    for row in pooled:
        nz = np.nonzero(row)[0]
        if len(nz) > top_n:
            top_idx = nz[np.argpartition(-row[nz], top_n - 1)[:top_n]]
        else:
            top_idx = nz
        order = np.argsort(-row[top_idx])
        top_idx = top_idx[order]
        results.append((top_idx.astype(int).tolist(), row[top_idx].astype(float).tolist()))
    return results


In [ ]:
import scipy.sparse as sp

def build_sparse(texts, jsonl_path, npz_path, batch_size, top_n):
    if os.path.exists(jsonl_path) and os.path.exists(npz_path):
        print("sparse cache hit -> loading", npz_path)
        return sp.load_npz(npz_path), None

    rows, cols, data, nnz_counts, failures = [], [], [], [], 0
    with open(jsonl_path, "w") as fout:
        for s in tqdm(range(0, len(texts), batch_size), desc="sparse"):
            batch = texts[s:s + batch_size]
            try:
                encoded = splade_encode_batch(batch, top_n)
            except Exception as e:
                print("sparse batch failed at", s, ":", e)
                failures += len(batch)
                for j in range(len(batch)):
                    fout.write(json.dumps({"product_id": product_ids[s + j],
                                           "indices": [], "values": [],
                                           "error": str(e)}) + "\n")
                continue
            for j, (idx, val) in enumerate(encoded):
                i = s + j
                fout.write(json.dumps({"product_id": product_ids[i],
                                       "indices": idx, "values": val}) + "\n")
                nnz_counts.append(len(idx))
                rows.extend([i] * len(idx))
                cols.extend(idx)
                data.extend(val)

    matrix = sp.csr_matrix((data, (rows, cols)),
                           shape=(len(texts), VOCAB_SIZE), dtype=np.float32)
    sp.save_npz(npz_path, matrix)
    stats = {"avg_nnz": float(np.mean(nnz_counts)) if nnz_counts else 0.0,
             "min_nnz": int(np.min(nnz_counts)) if nnz_counts else 0,
             "max_nnz": int(np.max(nnz_counts)) if nnz_counts else 0,
             "failures": failures}
    return matrix, stats

product_sparse, SPARSE_STATS = build_sparse(
    product_texts, SPARSE_JSONL, SPARSE_INDEX, SPARSE_BATCH_SIZE, SPARSE_TOP_N)
if SPARSE_STATS is None:
    nnz = product_sparse.getnnz(axis=1)
    SPARSE_STATS = {"avg_nnz": float(nnz.mean()), "min_nnz": int(nnz.min()),
                    "max_nnz": int(nnz.max()), "failures": 0}
print("sparse matrix:", product_sparse.shape, "| stats:", SPARSE_STATS)
assert 0 < SPARSE_STATS["avg_nnz"] < VOCAB_SIZE, "avg nnz must be >0 and < vocab"


## 4. Build indexes — Option B: FAISS dense + CSR sparse + manifest

In [ ]:
import faiss

# Dense: inner product over L2-normalized vectors == cosine similarity.
dense_index = faiss.IndexFlatIP(DENSE_DIM)
dense_index.add(np.ascontiguousarray(product_dense, dtype=np.float32))
faiss.write_index(dense_index, DENSE_INDEX)

print("dense index:", dense_index.ntotal, "vectors, dim", DENSE_DIM)

# Sparse index is already saved as CSR .npz. Search = dot product.
def sparse_search(query_indices, query_values, top_k=10):
    q = np.zeros(VOCAB_SIZE, dtype=np.float32)
    q[query_indices] = query_values
    scores = np.asarray(product_sparse.dot(q)).ravel()
    if top_k >= len(scores):
        top = np.argsort(-scores)
    else:
        top = np.argpartition(-scores, top_k)[:top_k]
        top = top[np.argsort(-scores[top])]
    return [(int(i), float(scores[i])) for i in top]

def dense_search(query_vec, top_k=10):
    q = query_vec.reshape(1, -1).astype(np.float32)
    if DENSE_NORMALIZE:
        norm = np.linalg.norm(q, axis=1, keepdims=True)
        norm[norm == 0] = 1.0
        q = q / norm
    scores, idx = dense_index.search(q, top_k)
    return [(int(i), float(s)) for i, s in zip(idx[0], scores[0]) if i != -1]


In [ ]:
manifest = {
    "num_products": int(len(catalog)),
    "dense_model": DENSE_MODEL,
    "dense_input_format": "multimodal_dict_text_image",
    "dense_embedding_policy": "single_vector_from_combined_text_image_input",
    "manual_text_image_mixing": False,
    "manual_text_image_vector_mixing": False,
    "dense_encode_method": "encode_document" if hasattr(dense_model, "encode_document") else "encode",
    "dense_load_config": DENSE_LOAD_CFG,
    "dense_batch_size": DENSE_BATCH_SIZE,
    "sparse_model": SPARSE_MODEL,
    "sparse_max_len": SPARSE_MAX_LEN,
    "dense_dimension": DENSE_DIM,
    "sparse_top_n": SPARSE_TOP_N,
    "sparse_vocab_size": VOCAB_SIZE,
    "vector_store": "faiss_plus_csr",
    "similarity": "cosine_via_inner_product_on_normalized_vectors",
    "id_mapping_file": IDS_TXT,
    "runtime": GPU_NAME,
    "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(MANIFEST, "w") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))


## 5. Sanity-check searches 

Self-retrieval probes: a product's own image / text / title should rank the
product itself at (or very near) the top.

In [ ]:
SANITY_TOP_K = 10

def encode_query_dense(q):
    fn = getattr(dense_model, "encode_query", None)
    kw = dict(convert_to_numpy=True, normalize_embeddings=DENSE_NORMALIZE)
    encode_fn = fn if fn is not None else dense_model.encode
    vec = encode_fn([q], **kw)
    return np.asarray(vec, dtype=np.float32)[0]

def _self_rank(res, target):
    return next((k for k, (i, _) in enumerate(res) if i == target), None)

SANITY = []
probe_rows = [0, len(catalog) // 2, len(catalog) - 1]

# (a) dense image-only self-retrieval
r = probe_rows[0]
res = dense_search(encode_query_dense(image_paths[r]), top_k=SANITY_TOP_K)
SANITY.append({"kind": "DENSE image-query", "product": product_ids[r],
               "rank": _self_rank(res, r),
               "top": [product_ids[i] for i, _ in res][:3]})

# (b) dense text-only self-retrieval
r = probe_rows[1]
res = dense_search(encode_query_dense(product_texts[r]), top_k=SANITY_TOP_K)
SANITY.append({"kind": "DENSE text-query", "product": product_ids[r],
               "rank": _self_rank(res, r),
               "top": [product_ids[i] for i, _ in res][:3]})

# (c) dense image+text combined query
r = probe_rows[2]
q = encode_query_dense({"text": catalog.iloc[r]["title"], "image": image_paths[r]})
res = dense_search(q, top_k=SANITY_TOP_K)
SANITY.append({"kind": "DENSE image+text", "product": product_ids[r],
               "rank": _self_rank(res, r),
               "top": [product_ids[i] for i, _ in res][:3]})

# (d) sparse title self-retrieval for each probe
for r in probe_rows:
    idx, val = splade_encode_batch([catalog.iloc[r]["title"]], SPARSE_TOP_N)[0]
    res = sparse_search(idx, val, top_k=SANITY_TOP_K)
    SANITY.append({"kind": "SPARSE title-query", "product": product_ids[r],
                   "rank": _self_rank(res, r),
                   "top": [product_ids[i] for i, _ in res][:3]})

for s in SANITY:
    print(f"{s['kind']} (product {s['product']}): "
          f"self rank={s['rank']}, top={s['top']}")


## 6. Write the indexing summary report 

In [ ]:
# TODO: write a short indexing report from the variables created above.
# Include the dense model, sparse model, vector dimensions, sparse nnz stats,
# index files, ID mapping, and the sanity-check retrieval results.
lines = [
    "# Stage 1 - Indexing Summary",
    "",
    "## Dense embedding",
    f"- Model: `{DENSE_MODEL}` (encode method="
    f"{'encode_document' if hasattr(dense_model, 'encode_document') else 'encode'}, device={DEVICE})",
    "- Input format: combined {\"text\": product_text, \"image\": image_path} dict, "
    "one forward pass per product -> one vector (no separate encode + averaging)",
    f"- Dimension: {DENSE_DIM}, batch size: {DENSE_BATCH_SIZE}, "
    f"wall time: {DENSE_MINUTES:.1f} min for {len(catalog)} products",
    f"- Vector norm mean/min/max: {_norms.mean():.4f} / {_norms.min():.4f} / {_norms.max():.4f}",
    "",
    "## Sparse embedding",
    f"- Model: `{SPARSE_MODEL}` (SPLADE++-style masked LM, log(1+relu) activations, max pooling)",
    f"- Vocab size: {VOCAB_SIZE}, top-n kept per product: {SPARSE_TOP_N}",
    f"- Stats: {SPARSE_STATS}",
    "",
    "## Indexes",
    f"- Dense: FAISS IndexFlatIP ({dense_index.ntotal} vectors, dim {DENSE_DIM}); "
    "cosine similarity via inner product on L2-normalized vectors",
    f"- Sparse: SciPy CSR matrix (shape {product_sparse.shape}); dot-product similarity",
    f"- ID mapping file: {IDS_TXT} (row i <-> line i)",
    "",
    "## Sanity-check searches",
] + [f"- {s['kind']} (product {s['product']}): self rank={s['rank']}, top={s['top']}"
     for s in SANITY] + [""]
with open(SUMMARY, "w") as f:
    f.write("\n".join(lines))
print("wrote", SUMMARY)
print("\n".join(lines))


## 7. Acceptance tests 

In [ ]:
ok = True
def check(name, cond):
    global ok
    ok = ok and bool(cond)
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}")

def _by_kind(k):
    return [s for s in SANITY if s["kind"] == k]

def _in_top5(records):
    return any(r["rank"] is not None and r["rank"] < 5 for r in records)

ids_on_disk = open(IDS_TXT).read().split()

check("dense vector count == catalog size", product_dense.shape[0] == len(catalog))
check("product_ids.txt aligns row-for-row with catalog", ids_on_disk == product_ids)
check("dense vectors L2-normalized (norm ~ 1.0)",
      bool(np.allclose(np.linalg.norm(product_dense, axis=1), 1.0, atol=1e-2)))
check("sparse avg nnz in (0, vocab_size)", 0 < SPARSE_STATS["avg_nnz"] < VOCAB_SIZE)
check("dense self-retrieval in top-5",
      _in_top5(_by_kind("DENSE image-query")) or _in_top5(_by_kind("DENSE text-query")))
check("sparse self-retrieval in top-5", _in_top5(_by_kind("SPARSE title-query")))
for p in (DENSE_NPY, IDS_TXT, SPARSE_JSONL, DENSE_INDEX, SPARSE_INDEX, MANIFEST, SUMMARY):
    check(f"exists: {p}", os.path.exists(p))

print("\nALL PASSED" if ok else "\nSOME CHECKS FAILED - see above")


## 8. Visualize the sanity / acceptance retrieval results

For each the Stage 1 retrieval retrieval check, show the **query** (image and/or text) next to the
**top-5 retrieved product images**. When the query is a product's own
image/text/title, that product's tile is outlined in **green** if it comes back
in the top-5 (the "returns that product near the top" criterion, made visual).

> Run this after the rest of the notebook — it reuses `dense_search`,
> `sparse_search`, `encode_query_dense`, `splade_encode_batch`, `catalog`,
> `image_paths`, and `product_ids` that are already in memory.

In [ ]:
import textwrap
import matplotlib.pyplot as plt
from PIL import Image

def _load_img(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return None

def _border(ax, color, width=3.5):
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_color(color); sp.set_linewidth(width)

def visualize(title, results, query_image=None, query_text=None,
              self_row=None, k=5):
    results = results[:k]
    n = len(results) + 1
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.2))
    ax0 = axes[0]
    if query_image is not None:
        img = _load_img(query_image)
        if img is not None:
            ax0.imshow(img)
    if query_text:
        ax0.set_title("\n".join(textwrap.wrap(f"QUERY: {query_text}", 20)), fontsize=8)
    _border(ax0, "blue")
    for ax, (row, score) in zip(axes[1:], results):
        img = _load_img(image_paths[row])
        if img is not None:
            ax.imshow(img)
        is_self = self_row is not None and row == self_row
        ax.set_title(f"{product_ids[row]}\nscore={score:.3f}", fontsize=7)
        _border(ax, "green" if is_self else "gray", 3.5 if is_self else 1.0)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

print("visualize() ready")


In [ ]:
probe_rows = [0, len(catalog) // 2, len(catalog) - 1]

r = probe_rows[0]
res = dense_search(encode_query_dense(image_paths[r]), top_k=5)
visualize("DENSE image-query", res, query_image=image_paths[r], self_row=r)

r = probe_rows[1]
res = dense_search(encode_query_dense(product_texts[r]), top_k=5)
visualize("DENSE text-query", res, query_text=product_texts[r][:80], self_row=r)

r = probe_rows[2]
res = dense_search(encode_query_dense({"text": catalog.iloc[r]["title"], "image": image_paths[r]}), top_k=5)
visualize("DENSE image+text query", res, query_image=image_paths[r],
          query_text=catalog.iloc[r]["title"], self_row=r)

r = probe_rows[0]
idx, val = splade_encode_batch([catalog.iloc[r]["title"]], SPARSE_TOP_N)[0]
res = sparse_search(idx, val, top_k=5)
visualize("SPARSE title-query", res, query_text=catalog.iloc[r]["title"], self_row=r)


In [ ]:
DEMO_QUERY = "gray fabric sofa for a small living room, not leather"

dense_res = dense_search(encode_query_dense(DEMO_QUERY), top_k=5)
visualize("DENSE free-text demo", dense_res, query_text=DEMO_QUERY)

idx, val = splade_encode_batch([DEMO_QUERY], SPARSE_TOP_N)[0]
sparse_res = sparse_search(idx, val, top_k=5)
visualize("SPARSE free-text demo", sparse_res, query_text=DEMO_QUERY)


# Stage 2 + 3


# Stage 2 + Stage 3 — Hybrid Retrieval & Cross-Encoder Rerank (inline, self-contained)

This notebook runs Stages 2 and 3 of the pipeline end-to-end on a single
Colab GPU. All code is inlined in cells (not subprocesses) so you see:

* Hugging Face model download progress (widgets)
* per-query retrieval progress (tqdm bars)
* per-query rerank progress (tqdm bars)
* any errors with full Python tracebacks

**Inputs (already produced by Stage 0 + Stage 1):**

* `data/catalog_subset.parquet`
* `data/queries.jsonl`  (20 queries: 10 text, 5 image, 5 image+text)
* `artifacts/embeddings/product_dense.npy`
* `artifacts/embeddings/product_ids.txt`
* `artifacts/index/dense_index.faiss`
* `artifacts/index/sparse_index.npz`
* `artifacts/index/index_manifest.json`

**Outputs (after running this notebook):**

* `outputs/retrieval_results.jsonl`
* `outputs/reranked_results.jsonl`
* `reports/retrieval_summary.md`
* `reports/reranking_summary.md`

**Where to put files on Drive:**

```
MyDrive/AIR_Phase3/
  project/                       <- the whole project folder (Drive upload)
    data/catalog_subset.parquet
    data/queries.jsonl
    artifacts/...
  runs/2026.../                  <- outputs land here
```

**Rerank backend:** this notebook uses the **local** `BAAI/bge-reranker-v2-m3`
cross-encoder on GPU (~2.3 GB download). To switch to the OpenRouter-based
rerank (no model download), see the note in cell 14.



In [ ]:
# Confirm the Drive paths and run folder before starting Stage 2/3.
import os, sys, json, time, subprocess, getpass, zipfile, shutil
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import output, drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab; using local paths.")

DRIVE_BASE = '/content/drive/MyDrive/AIR_Phase3'
DRIVE_PROJECT_DIR = f'{DRIVE_BASE}/project'
UPLOAD_ZIP_NAMES = [
    'AIR_Phase3_student_minimal.zip',
    'AIR_Phase3_student_runnable_compact.zip',
    'AIR_Phase3_upload.zip',
]


def _looks_like_project(path):
    return os.path.isfile(os.path.join(path, 'pyproject.toml')) and os.path.isdir(os.path.join(path, 'src'))

if IN_COLAB and not _looks_like_project(DRIVE_PROJECT_DIR):
    os.makedirs(DRIVE_BASE, exist_ok=True)
    upload_zip = next((os.path.join(DRIVE_BASE, name) for name in UPLOAD_ZIP_NAMES
                       if os.path.isfile(os.path.join(DRIVE_BASE, name))), None)
    if upload_zip:
        print(f"Found {upload_zip}, extracting...")
        with zipfile.ZipFile(upload_zip) as zf:
            zf.extractall(DRIVE_BASE)
    else:
        raise SystemExit(
            f"Project folder not found. Upload one of {UPLOAD_ZIP_NAMES} "
            f"to MyDrive/AIR_Phase3/ first.")

if IN_COLAB:
    for candidate in [DRIVE_PROJECT_DIR, f'{DRIVE_BASE}/projectf', DRIVE_BASE]:
        if _looks_like_project(candidate):
            DRIVE_PROJECT_DIR = candidate
            break
    if not _looks_like_project(DRIVE_PROJECT_DIR):
        raise SystemExit("Could not find pyproject.toml and src/ after extracting the upload zip.")
else:
    DRIVE_PROJECT_DIR = os.getcwd()

# Symlink target on local Colab disk for fast I/O.
WORK = '/content/project'
# Where outputs land on Drive.
RUN_TS = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_RUN_DIR = f'{DRIVE_BASE}/runs/{RUN_TS}'

print(f"DRIVE_BASE        = {DRIVE_BASE}")
print(f"DRIVE_PROJECT_DIR = {DRIVE_PROJECT_DIR}")
print(f"WORK              = {WORK}")
print(f"DRIVE_RUN_DIR     = {DRIVE_RUN_DIR}")
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(f"GPU: {gpu}")


In [ ]:
# Install the retrieval and reranking runtime from pyproject.toml.
import os
import subprocess
import sys

try:
    import tomllib
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tomli"])
    import tomli as tomllib

PROJECT_FILE = os.path.join(DRIVE_PROJECT_DIR, "pyproject.toml") if "DRIVE_PROJECT_DIR" in globals() else "pyproject.toml"
with open(PROJECT_FILE, "rb") as f:
    project_cfg = tomllib.load(f)

deps = list(project_cfg["project"].get("dependencies", []))
deps += list(project_cfg["project"].get("optional-dependencies", {}).get("stage1-gpu", []))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *deps])
print("dependencies installed from", PROJECT_FILE)
print("If Colab asks for a runtime restart, restart and rerun from the setup cell.")


In [ ]:
# TODO: verify the runtime versions after restarting the notebook kernel.
# --- 2b. Verify versions after the kernel restart above ---
import torch, faiss, sentence_transformers, transformers
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print("faiss", faiss.__version__)
print("sentence-transformers", sentence_transformers.__version__)
print("transformers", transformers.__version__)
import PIL._typing; print("Pillow", PIL.__version__, "_typing ok")



In [ ]:
# TODO: link the Drive project into local Colab storage for faster I/O.
# --- 3. Symlink project from Drive to local /content ---
import os

if not os.path.isdir(DRIVE_PROJECT_DIR):
    raise SystemExit(
        f"{DRIVE_PROJECT_DIR} not found. Upload the project folder to "
        f"MyDrive/AIR_Phase3/project first.")

if os.path.islink(WORK):
    os.unlink(WORK)
elif os.path.isdir(WORK):
    import shutil; shutil.rmtree(WORK)
os.symlink(DRIVE_PROJECT_DIR, WORK)
os.chdir(WORK)
print("project at:", WORK)
print("top-level:", sorted(os.listdir(WORK))[:15])



In [ ]:
# TODO: check that all Stage 0/1 artifacts are present before retrieval.
# --- 4. Verify Stage 0+1 artifacts are present ---
import os
need = [
    'data/catalog_subset.parquet',
    'data/queries.jsonl',
    'artifacts/embeddings/product_dense.npy',
    'artifacts/embeddings/product_ids.txt',
    'artifacts/embeddings/product_sparse.jsonl',
    'artifacts/index/dense_index.faiss',
    'artifacts/index/sparse_index.npz',
    'artifacts/index/index_manifest.json',
]
missing = [p for p in need if not os.path.exists(p)]
if missing:
    raise SystemExit("MISSING ARTIFACTS: " + ', '.join(missing))
print("all", len(need), "artifacts present")

import json
qs = [json.loads(l) for l in open('data/queries.jsonl', encoding='utf-8') if l.strip()]
print(f"queries: {len(qs)} "
      f"(text={sum(1 for q in qs if q['query_type']=='text')}, "
      f"image={sum(1 for q in qs if q['query_type']=='image')}, "
      f"image_text={sum(1 for q in qs if q['query_type']=='image_text')})")



In [ ]:
# --- 5. Imports + constants ---
import collections
import json
import os
import time

import numpy as np
import pandas as pd
import scipy.sparse as sp
import faiss
from tqdm.auto import tqdm

# Values are read from the Stage 1 index manifest so they always match the
# artifacts actually produced, with sensible retrieval-time defaults.
_manifest = json.load(open("artifacts/index/index_manifest.json"))
DENSE_MODEL  = _manifest["dense_model"]
SPARSE_MODEL = _manifest["sparse_model"]
SPARSE_VOCAB = _manifest["sparse_vocab_size"]
SPARSE_TOP_N = _manifest["sparse_top_n"]
DENSE_DIM    = _manifest["dense_dimension"]
RRF_K        = 60
W_DENSE      = 0.55
W_SPARSE     = 0.45
TOP_K_DENSE  = 100
TOP_K_SPARSE = 100
PREFUSION    = "rrf"

OUT_DIR = "outputs"
EMB_DIR = "artifacts/embeddings"
INDEX_DIR = "artifacts/index"
DENSE_CACHE  = "artifacts/query_cache/dense_queries.npz"
SPARSE_CACHE = "artifacts/query_cache/sparse_queries.jsonl"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("artifacts/query_cache", exist_ok=True)
os.makedirs("reports", exist_ok=True)

print("OK")


In [ ]:
# TODO: load the catalog, indexes, ID mapping, and queries used by retrieval.
# --- 6. Load catalog + indexes + queries ---
print("[retrieve] loading catalog data/catalog_subset.parquet")
catalog = pd.read_parquet("data/catalog_subset.parquet")
# Resolve image_path relative to project root
catalog["abs_image_path"] = catalog["image_path"].map(
    lambda p: p if os.path.isabs(p) else os.path.normpath(os.path.join("/content/project", p))
)
print(f"catalog: {len(catalog)} rows")

print("[retrieve] loading FAISS index from artifacts/index")
dense_index = faiss.read_index(os.path.join(INDEX_DIR, "dense_index.faiss"))
print("[retrieve] loading CSR sparse index from artifacts/index")
sparse_index = sp.load_npz(os.path.join(INDEX_DIR, "sparse_index.npz"))
with open(os.path.join(EMB_DIR, "product_ids.txt"), encoding="utf-8") as f:
    product_ids = [l.strip() for l in f if l.strip()]
print(f"dense: {dense_index.ntotal} vectors, d={dense_index.d}; "
      f"sparse: {sparse_index.shape}, nnz={sparse_index.nnz}; ids: {len(product_ids)}")

print("[retrieve] loading queries data/queries.jsonl")
queries = []
with open("data/queries.jsonl", encoding="utf-8") as f:
    for ln, line in enumerate(f, 1):
        line = line.strip()
        if not line: continue
        q = json.loads(line)
        for k in ("query_id", "query_text", "query_image_path", "query_type", "split"):
            if k not in q:
                raise ValueError(f"line {ln} missing '{k}'")
        if q["query_type"] not in ("text", "image", "image_text"):
            raise ValueError(f"line {ln}: bad query_type {q['query_type']!r}")
        if q["query_type"] in ("text", "image_text") and not q["query_text"]:
            raise ValueError(f"line {ln}: text/image_text query needs query_text")
        if q["query_type"] in ("image", "image_text") and not q["query_image_path"]:
            raise ValueError(f"line {ln}: image/image_text query needs query_image_path")
        if q["query_image_path"]:
            rel = q["query_image_path"]
            abs_p = rel if os.path.isabs(rel) else os.path.join("/content/project", rel)
            abs_p = os.path.normpath(abs_p)
            q["abs_image_path"] = abs_p
        queries.append(q)
print(f"queries: {len(queries)}; by type "
      f"{dict(collections.Counter(q['query_type'] for q in queries))}")



In [ ]:
# --- 7. Load Qwen3-VL-Embedding-2B on GPU ---
# This cell downloads the model the first time it runs.
import torch
from sentence_transformers import SentenceTransformer

print(f"loading dense model {DENSE_MODEL} on cuda")

dense_model = SentenceTransformer(DENSE_MODEL, trust_remote_code=True, device="cuda")

print("loaded:", DENSE_MODEL)


In [ ]:
# --- 8. Load SPLADE++ sparse encoder on GPU ---
# Downloads prithivida/Splade_PP_en_v2 (~440 MB) the first time.
from transformers import AutoTokenizer, AutoModelForMaskedLM

print(f"loading sparse model {SPARSE_MODEL} on cuda")
sp_tokenizer = AutoTokenizer.from_pretrained(SPARSE_MODEL)
sp_model = AutoModelForMaskedLM.from_pretrained(SPARSE_MODEL).to("cuda").eval()
print("loaded:", SPARSE_MODEL, "| vocab:", sp_model.config.vocab_size)

@torch.no_grad()
def splade_encode_batch(texts, top_n=SPARSE_TOP_N, max_len=256):
    enc = sp_tokenizer(texts, padding=True, truncation=True,
                       max_length=max_len, return_tensors="pt").to("cuda")

    logits = sp_model(**enc).logits
    activations = torch.log1p(torch.relu(logits))
    mask = enc["attention_mask"].unsqueeze(-1)
    pooled, _ = (activations * mask).max(dim=1)
    pooled = pooled.cpu().numpy()
    results = []
    for row in pooled:
        nz = np.nonzero(row)[0]
        if len(nz) > top_n:
            top_idx = nz[np.argpartition(-row[nz], top_n - 1)[:top_n]]
        else:
            top_idx = nz
        order = np.argsort(-row[top_idx])
        top_idx = top_idx[order]
        results.append((top_idx.astype(int).tolist(), row[top_idx].astype(float).tolist()))
    return results

print("splade_encode_batch ready")


In [ ]:
# TODO: decide how this run should handle image queries with missing files.
# --- 8b. Patch missing image paths: skip image-bearing queries whose file is missing ---
import os

queries_orig = list(queries)
queries = []
skipped = []
for q in queries_orig:
    if q.get('query_image_path'):
        rel = q['query_image_path']
        abs_p = rel if os.path.isabs(rel) else os.path.join('/content/project', rel)
        abs_p = os.path.normpath(abs_p)
        if not os.path.exists(abs_p):
            print(f"  SKIP {q['query_id']} ({q['query_type']}): missing {abs_p}")
            skipped.append(q['query_id'])
            continue
    queries.append(q)
print()
print(f"Will process {len(queries)}/{len(queries_orig)} queries (skipped: {skipped})")
print("(the skipped queries are dropped from this run, not replaced)")

In [ ]:
# --- 9. Encode dense query vectors (cached) ---
from PIL import Image

cached = {}
if os.path.isfile(DENSE_CACHE):
    with np.load(DENSE_CACHE, allow_pickle=False) as z:
        for k in z.files:
            cached[k] = z[k]
    print(f"dense query cache hit: {len(cached)} vectors")

def _load_image(path):
    """Load image as PIL.Image so Qwen3-VL doesn't try to base64 the path."""
    return Image.open(path).convert("RGB")

def _dense_input_for_query(q):
    if q["query_type"] == "text":
        return q["query_text"]
    if q["query_type"] == "image":
        return _load_image(q["abs_image_path"])
    return {"text": q["query_text"], "image": _load_image(q["abs_image_path"])}

def _dense_encode_one(q):
    fn = getattr(dense_model, "encode_query", None)
    inp = _dense_input_for_query(q)
    kw = dict(convert_to_numpy=True, normalize_embeddings=True)

    encode_fn = fn if fn is not None else dense_model.encode
    vec = encode_fn([inp], **kw)
    return np.asarray(vec, dtype=np.float32)[0]

todo = [q for q in queries if q["query_id"] not in cached]
print(f"encoding {len(todo)} dense query vectors ({len(cached)} cached)")
out = {}
for q in tqdm(todo, desc="dense query encode"):
    out[q["query_id"]] = _dense_encode_one(q)
all_dense = {**cached, **out}
if out:
    os.makedirs(os.path.dirname(DENSE_CACHE), exist_ok=True)
    np.savez(DENSE_CACHE, **all_dense)
    print(f"saved {len(all_dense)} dense query vectors to {DENSE_CACHE}")
print("dense query vectors:", len(all_dense))


In [ ]:
# --- 10. Encode sparse query vectors (cached) ---
cached = {}
if os.path.isfile(SPARSE_CACHE):
    with open(SPARSE_CACHE, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec.get("indices"):
                cached[rec["query_id"]] = (
                    np.asarray(rec["indices"], dtype=np.int32),
                    np.asarray(rec["values"], dtype=np.float32),
                )
    print(f"sparse query cache hit: {len(cached)} vectors")

todo = [q for q in queries
        if q["query_type"] in ("text", "image_text")
        and q["query_id"] not in cached]
print(f"encoding {len(todo)} sparse query vectors")
BATCH = 16
new_records = []
for s in tqdm(range(0, len(todo), BATCH), desc="sparse query encode"):
    batch_qs = todo[s:s+BATCH]
    batch = [q["query_text"] for q in batch_qs]
    encoded = splade_encode_batch(batch)
    for q, (idx, val) in zip(batch_qs, encoded):
        cached[q["query_id"]] = (np.asarray(idx, dtype=np.int32), np.asarray(val, dtype=np.float32))
        new_records.append({"query_id": q["query_id"], "indices": idx, "values": val})

if new_records:
    with open(SPARSE_CACHE, "a", encoding="utf-8") as f:
        for rec in new_records:
            f.write(json.dumps(rec) + "\n")
    print(f"appended {len(new_records)} sparse query vectors to {SPARSE_CACHE}")
print("sparse query vectors:", len(cached))


In [ ]:
# --- 11. Hybrid search + RRF/weighted prefusion + write outputs ---
def normalize_scores(pairs):
    if not pairs: return {}
    vals = np.array([s for _, s in pairs], dtype=np.float32)
    lo, hi = float(vals.min()), float(vals.max())
    if hi - lo < 1e-9:
        return {i: 0.0 for i, _ in pairs}
    return {i: float((s - lo) / (hi - lo)) for i, s in pairs}

def dense_search(di, q_vec, top_k):
    q = q_vec.reshape(1, -1).astype(np.float32)
    scores, idx = di.search(q, top_k)
    return [(int(i), float(s)) for i, s in zip(idx[0], scores[0]) if i != -1]

def sparse_search(si, q_idx, q_val, top_k):
    if q_idx is None or len(q_idx) == 0: return []
    v = np.zeros(si.shape[1], dtype=np.float32)
    v[np.asarray(q_idx, dtype=np.int64)] = np.asarray(q_val, dtype=np.float32)
    scores = np.asarray(si.dot(v)).ravel()
    if top_k >= len(scores):
        top = np.argsort(-scores)
    else:
        top = np.argpartition(-scores, top_k)[:top_k]
        top = top[np.argsort(-scores[top])]
    return [(int(i), float(scores[i])) for i in top]

def merge_and_score(d_hits, s_hits, query_type, prefusion_method):
    sparse_applicable = query_type in ("text", "image_text")
    d_norm = normalize_scores(d_hits)
    s_norm = normalize_scores(s_hits) if sparse_applicable else {}
    d_rank = {i: r for r, (i, _) in enumerate(d_hits)}
    s_rank = {i: r for r, (i, _) in enumerate(s_hits)}
    d_score_map = {i: s for i, s in d_hits}
    s_score_map = {i: s for i, s in s_hits}
    candidates = set(d_score_map) | set(s_score_map)

    fused = {}
    if prefusion_method == "rrf":
        for i in candidates:
            score = 0.0
            if i in d_rank:
                score += 1.0 / (RRF_K + d_rank[i] + 1)
            if sparse_applicable and i in s_rank:
                score += 1.0 / (RRF_K + s_rank[i] + 1)
            fused[i] = score
    else:
        w_d, w_s = (W_DENSE, W_SPARSE) if sparse_applicable else (1.0, 0.0)
        for i in candidates:
            fused[i] = w_d * d_norm.get(i, 0.0) + w_s * s_norm.get(i, 0.0)

    rows = []
    for i, pf in sorted(fused.items(), key=lambda kv: -kv[1]):
        source = []
        if i in d_score_map:
            source.append("dense")
        if i in s_score_map:
            source.append("sparse")
        rows.append((i, d_score_map.get(i), s_score_map.get(i), pf, source))
    return rows

out_records = []
n_unique_cands, dense_n, sparse_n, overlap = [], 0, 0, 0
t0 = time.time()
for q in tqdm(queries, desc="retrieval"):
    if q["query_id"] in all_dense:
        d_hits = dense_search(dense_index, all_dense[q["query_id"]], TOP_K_DENSE)
    else:
        d_hits = []
    s_hits = []
    if q["query_type"] in ("text", "image_text") and q["query_id"] in cached:
        qidx, qval = cached[q["query_id"]]
        s_hits = sparse_search(sparse_index, qidx, qval, TOP_K_SPARSE)
    merged = merge_and_score(d_hits, s_hits, q["query_type"], PREFUSION)
    sparse_applicable = q["query_type"] in ("text", "image_text")
    results = []
    d_set = {i for i, _ in d_hits}
    s_set = {i for i, _ in s_hits}
    for rank, (i, d_s, s_s, pf_s, src) in enumerate(merged, 1):
        results.append({
            "rank": rank,
            "product_id": product_ids[i],
            "dense_score": d_s,
            "sparse_score": s_s if sparse_applicable else None,
            "prefusion_score": pf_s,
            "source": src,
        })
    out_records.append({
        "query_id": q["query_id"],
        "query_type": q["query_type"],
        "run_name": "hybrid_dense_sparse_prefusion",
        "sparse_applicable": sparse_applicable,
        "results": results,
    })
    dense_n += len(d_hits); sparse_n += len(s_hits)
    overlap += len(d_set & s_set)
    n_unique_cands.append(len(merged))

print(f"\nretrieval done in {time.time()-t0:.1f}s | "
      f"mean unique cands/query={np.mean(n_unique_cands):.1f} | "
      f"mean dense-hits={dense_n/len(queries):.1f} | "
      f"mean sparse-hits={sparse_n/len(queries):.1f} | "
      f"mean overlap={overlap/len(queries):.1f}")

with open(f"{OUT_DIR}/retrieval_results.jsonl", "w", encoding="utf-8") as f:
    for r in out_records:
        f.write(json.dumps(r) + "\n")
print(f"wrote {len(out_records)} lines to outputs/retrieval_results.jsonl")

by_type = collections.Counter(q["query_type"] for q in queries)
n_text, n_image, n_image_text = by_type.get("text", 0), by_type.get("image", 0), by_type.get("image_text", 0)
examples = []
for r in out_records[:3]:
    top3 = r["results"][:3]
    examples.append(f"- **{r['query_id']}** ({r['query_type']}) -> " + ", ".join(
        f"#{x['rank']} {x['product_id']} (pf={x['prefusion_score']:.3f})" for x in top3))
md = [
    "# Stage 2 - Retrieval Summary", "",
    f"- Queries by type: text={n_text}, image={n_image}, image_text={n_image_text} (total={len(queries)})",
    f"- Dense top-K: {TOP_K_DENSE} | Sparse top-K: {TOP_K_SPARSE}",
    f"- Prefusion method: **{PREFUSION}** (RRF k={RRF_K} | weighted w_dense={W_DENSE}, w_sparse={W_SPARSE})",
    f"- Average unique candidates per query: **{np.mean(n_unique_cands):.1f}**",
    f"- Average candidates in BOTH dense and sparse: **{overlap/len(queries):.1f}**", "",
    "## Top results for the first 3 queries", "", *examples, "",
]
with open("reports/retrieval_summary.md", "w", encoding="utf-8") as f:
    f.write("\n".join(md))
print("wrote reports/retrieval_summary.md")


In [ ]:
# TODO: run and review the Stage 2 schema checks after writing retrieval output.
# --- 12. Stage 2 schema sanity tests (inline) ---
import json, os
n_pass = n_fail = 0
def check(name, cond):
    global n_pass, n_fail
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}")
    if cond: n_pass += 1
    else: n_fail += 1

records = [json.loads(l) for l in open("outputs/retrieval_results.jsonl", encoding="utf-8") if l.strip()]
check("retrieval_results.jsonl non-empty", bool(records))
catalog = pd.read_parquet("data/catalog_subset.parquet")
catalog_ids = set(catalog["product_id"].tolist())
queries_by_id = {q["query_id"]: q for q in queries}
for r in records:
    check(f"{r['query_id']} in queries", r["query_id"] in queries_by_id)
    check(f"{r['query_id']} run_name is hybrid", r.get("run_name") == "hybrid_dense_sparse_prefusion")
    check(f"{r['query_id']} sparse_applicable matches",
          r.get("sparse_applicable") == (queries_by_id[r["query_id"]]["query_type"] != "image"))
    for h in r.get("results", []):
        check(f"{r['query_id']} hit {h['product_id']} in catalog", h["product_id"] in catalog_ids)
        check(f"{r['query_id']} hit has prefusion_score", "prefusion_score" in h)
        check(f"{r['query_id']} hit has source", "source" in h)
        check(f"{r['query_id']} hit rank is int", isinstance(h.get("rank"), int))
print(f"\n{n_pass} passed, {n_fail} failed")



In [ ]:
# --- 13. Load BGE cross-encoder reranker ---
# This model scores each (query_text, product_text) pair.
from sentence_transformers import CrossEncoder

CE_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
print(f"loading cross-encoder {CE_MODEL_NAME} on cuda")

ce_model = CrossEncoder(CE_MODEL_NAME, trust_remote_code=True, device="cuda", max_length=512)

print("loaded:", CE_MODEL_NAME)


In [ ]:
# --- 14. Build the (query_text, product_text) rerank input pairs ---
# Uses product metadata fields: title, product_type, category, brand, color,
# material, style, bullet points, description.
import math

cat_by_id = {r.product_id: r for r in catalog.itertuples()}

def _safe(v, default=''):
    """Return v as string; treat None and NaN as empty."""
    if v is None: return default
    if isinstance(v, float) and math.isnan(v): return default
    return v

def build_product_text(row, max_chars=600):
    fields = {
        "Title": _safe(getattr(row, 'title', None)),
        "Category": _safe(getattr(row, 'category_path', None)),
        "Product type": _safe(getattr(row, 'product_type', None)),
        "Brand": _safe(getattr(row, 'brand', None)),
        "Color": _safe(getattr(row, 'color', None)),
        "Material": _safe(getattr(row, 'material', None)),
        "Style": _safe(getattr(row, 'style', None)),
        "Features": _safe(getattr(row, 'bullet_points', None)),
        "Description": _safe(getattr(row, 'description', None)),
    }
    text = " | ".join(f"{k}: {v}" for k, v in fields.items() if v)
    return text[:max_chars]

def query_text_for(q):
    if q["query_type"] == "image":
        return "Find visually similar products."
    return q["query_text"]

RERANK_TOP_N = 100
print(f"building rerank input pairs (top {RERANK_TOP_N} per query)")
pairs_per_query = []
for r in tqdm(out_records, desc="build pairs"):
    q = queries_by_id[r["query_id"]]
    qtext = query_text_for(q)
    hits = r["results"][:RERANK_TOP_N]
    pairs = []
    for h in hits:
        row = cat_by_id.get(h["product_id"])
        if row is None: continue
        pairs.append((qtext, build_product_text(row)))
    pairs_per_query.append((r["query_id"], q["query_type"], pairs))
total = sum(len(p[2]) for p in pairs_per_query)
print(f"total (query, product) pairs to score: {total}")


In [ ]:
# --- 15. Score all pairs with BGE + write reranked_results.jsonl + summary ---
import math

def zscore(vals):
    if not vals: return []
    arr = np.array(vals, dtype=np.float32)
    mu, sd = float(arr.mean()), float(arr.std() + 1e-9)
    return [float((v - mu) / sd) for v in arr]

WEIGHTS = {
    "text": {"ce": 0.65, "dense": 0.20, "sparse": 0.15},
    "image": {"ce": 0.60, "dense": 0.40, "sparse": 0.00},
    "image_text": {"ce": 0.55, "dense": 0.30, "sparse": 0.15},
}

rerank_records = []
strategy_a_lines = []
t0 = time.time()
for qid, qtype, pairs in tqdm(pairs_per_query, desc="rerank"):
    q = queries_by_id[qid]
    retrieval = next(r for r in out_records if r["query_id"] == qid)
    hits = retrieval["results"][:RERANK_TOP_N]
    weights = WEIGHTS[qtype]

    if not pairs:
        rerank_records.append({"query_id": qid, "run_name": "hybrid_cross_encoder_fused",
                               "reranker_type": "multimodal_cross_encoder", "results": []})
        strategy_a_lines.append({"query_id": qid, "run_name": "hybrid_cross_encoder_ce_only",
                                 "reranker_type": "multimodal_cross_encoder", "results": []})
        continue

    ce_scores = np.asarray(ce_model.predict(pairs), dtype=np.float32).ravel()
    dense_scores = [h.get("dense_score") for h in hits]
    sparse_scores = [h.get("sparse_score") for h in hits]

    ce_z = zscore(list(ce_scores))
    dense_z = zscore([d if d is not None else 0.0 for d in dense_scores])
    sparse_z = zscore([s if s is not None else 0.0 for s in sparse_scores])

    fused_hits, strategy_a_hits = [], []
    for h, ce, cez, dz, sz in zip(hits, ce_scores, ce_z, dense_z, sparse_z):
        fh = dict(h)
        fh["cross_encoder_score"] = float(ce)
        fh["final_score"] = weights["ce"] * cez + weights["dense"] * dz + weights["sparse"] * sz
        fused_hits.append(fh)
        ah = dict(fh)
        ah["final_score"] = float(ce)
        strategy_a_hits.append(ah)

    fused_hits.sort(key=lambda h: -h["final_score"])
    for rank, h in enumerate(fused_hits, start=1):
        h["rank"] = rank
    strategy_a_hits.sort(key=lambda h: -h["final_score"])
    for rank, h in enumerate(strategy_a_hits, start=1):
        h["rank"] = rank

    rerank_records.append({"query_id": qid, "run_name": "hybrid_cross_encoder_fused",
                           "reranker_type": "multimodal_cross_encoder", "results": fused_hits})
    strategy_a_lines.append({"query_id": qid, "run_name": "hybrid_cross_encoder_ce_only",
                             "reranker_type": "multimodal_cross_encoder", "results": strategy_a_hits})

print(f"\nrerank done in {time.time()-t0:.1f}s")

with open(f"{OUT_DIR}/reranked_results.jsonl", "w", encoding="utf-8") as f:
    for r in rerank_records:
        f.write(json.dumps(r) + "\n")
print(f"wrote {len(rerank_records)} lines to outputs/reranked_results.jsonl")

with open(f"{OUT_DIR}/reranked_strategy_a.jsonl", "w", encoding="utf-8") as f:
    for r in strategy_a_lines:
        f.write(json.dumps(r) + "\n")
print(f"wrote strategy A ordering to outputs/reranked_strategy_a.jsonl")

for qt in ("text", "image", "image_text"):
    rows = [r for r in rerank_records if r["query_type"] == qt]
    if not rows: continue
    print(f"\n=== {qt}: top-1 of {len(rows)} queries ===")
    for r in rows[:3]:
        top1 = r["results"][0]
        q = queries_by_id[r["query_id"]]
        print(f"  {r['query_id']} -> #{top1['rank']} {top1['product_id']} "
              f"(final={top1['final_score']:.3f}, "
              f"q='{(q.get('query_text') or q.get('query_image_path',''))[:60]}')")


In [ ]:
# --- 16. Write reranking_summary.md ---
def find_by(qt, need_neg=False):
    for r in rerank_records:
        q = queries_by_id[r["query_id"]]
        if q["query_type"] != qt: continue
        if need_neg and " not " not in (q.get("query_text") or "").lower(): continue
        return r, q
    return None, None

r_neg, q_neg = find_by("text", need_neg=True)
r_it, q_it = find_by("image_text")

lines = [
    "# Stage 3 - Reranking Summary", "",
    f"- Reranker: `{CE_MODEL_NAME}` (multimodal cross-encoder)",
    "- Pair format: (query_text_or_dict, product_text) built from title/category/brand/"
    "color/material/style/bullets/description",
    f"- Reranking depth: {RERANK_TOP_N}",
    "- Score normalization: per-query z-score for cross-encoder, dense, and sparse scores",
    f"- Fusion weights: {json.dumps(WEIGHTS)}", "",
    "## Before/after examples", "",
]
for r in rerank_records[:3]:
    qid = r["query_id"]
    before = next(x for x in out_records if x["query_id"] == qid)["results"][:5]
    after = r["results"][:5]
    lines.append(f"**{qid}**: before={[h['product_id'] for h in before]} | "
                 f"after={[h['product_id'] for h in after]}")
lines.append("")
if q_neg:
    lines += ["## Negation example", f"Query {q_neg['query_id']}: {q_neg.get('query_text')}",
              f"Top after rerank: {[h['product_id'] for h in r_neg['results'][:5]]}", ""]
if q_it:
    lines += ["## Image+text example",
              f"Query {q_it['query_id']}: {q_it.get('query_text')} (image: {q_it.get('query_image_path')})",
              f"Top after rerank: {[h['product_id'] for h in r_it['results'][:5]]}", ""]

with open("reports/reranking_summary.md", "w") as f:
    f.write("\n".join(lines))
print("wrote reports/reranking_summary.md")


In [ ]:
# TODO: run and review the Stage 3 schema checks after writing rerank output.
# --- 17. Stage 3 schema sanity tests (inline) ---
import json, os
n_pass = n_fail = 0
def check(name, cond):
    global n_pass, n_fail
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}")
    if cond: n_pass += 1
    else: n_fail += 1

rr = [json.loads(l) for l in open("outputs/reranked_results.jsonl", encoding="utf-8") if l.strip()]
ret = {r["query_id"]: r for r in out_records}
catalog = pd.read_parquet("data/catalog_subset.parquet")
catalog_ids = set(catalog["product_id"].tolist())
for r in rr:
    check(f"{r['query_id']} exists in retrieval", r["query_id"] in ret)
    if r["query_id"] in ret:
        ret_ids = {h["product_id"] for h in ret[r["query_id"]]["results"]}
        rer_ids = {h["product_id"] for h in r["results"]}
        check(f"{r['query_id']} rerank is subset of retrieval", rer_ids.issubset(ret_ids))
    for h in r.get("results", []):
        check(f"{r['query_id']} hit {h['product_id']} in catalog", h["product_id"] in catalog_ids)
        check(f"{r['query_id']} hit has final_score", "final_score" in h)
        check(f"{r['query_id']} hit has cross_encoder_score", "cross_encoder_score" in h)
        check(f"{r['query_id']} hit has source", "source" in h)
        check(f"{r['query_id']} hit rank is int", isinstance(h.get("rank"), int))
print(f"\n{n_pass} passed, {n_fail} failed")



In [ ]:
# TODO: copy the finished outputs, reports, and query cache back to Drive.
# --- 18. Copy all outputs to your Google Drive ---
import shutil, os, json
os.makedirs(DRIVE_RUN_DIR + '/outputs', exist_ok=True)
os.makedirs(DRIVE_RUN_DIR + '/reports', exist_ok=True)
os.makedirs(DRIVE_RUN_DIR + '/artifacts/query_cache', exist_ok=True)

for fn in os.listdir('outputs'):
    shutil.copy('outputs/' + fn, DRIVE_RUN_DIR + '/outputs/' + fn)
for fn in os.listdir('reports'):
    shutil.copy('reports/' + fn, DRIVE_RUN_DIR + '/reports/' + fn)
if os.path.exists('artifacts/query_cache/dense_queries.npz'):
    shutil.copy('artifacts/query_cache/dense_queries.npz',
                DRIVE_RUN_DIR + '/artifacts/query_cache/dense_queries.npz')
if os.path.exists('artifacts/query_cache/sparse_queries.jsonl'):
    shutil.copy('artifacts/query_cache/sparse_queries.jsonl',
                DRIVE_RUN_DIR + '/artifacts/query_cache/sparse_queries.jsonl')

# Final EVAL.md
n_queries = sum(1 for _ in open('data/queries.jsonl', encoding='utf-8') if _.strip())
top1 = {}
with open('outputs/reranked_results.jsonl', encoding='utf-8') as f:
    for ln in f:
        r = json.loads(ln)
        if r['results']:
            top1.setdefault(r['query_type'], []).append(r['results'][0]['product_id'])

gpu_name = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
EVAL = [
    '# Stage 2 + Stage 3 run', '',
    'Run timestamp: ' + RUN_TS,
    'GPU:          ' + gpu_name,
    'Queries:      ' + str(n_queries), '',
    '## What ran',
    '- Stage 2 (hybrid retrieval: dense Qwen3-VL + sparse SPLADE, RRF prefusion)',
    '- Stage 3 (cross-encoder rerank with BGE-reranker-v2-m3, z-score fusion)', '',
    '## Top-1 by query type (reranked output)',
]
for qt, ids in top1.items():
    EVAL.append('- ' + qt + ': ' + str(len(ids)) + ' queries, top-1 -> ' + ids[0])
EVAL += ['', '## Files',
         '- outputs/retrieval_results.jsonl',
         '- outputs/reranked_results.jsonl',
         '- outputs/reranked_strategy_a.jsonl',
         '- reports/retrieval_summary.md',
         '- reports/reranking_summary.md']
with open(DRIVE_RUN_DIR + '/EVAL.md', 'w') as f:
    f.write('\n'.join(EVAL))

print('\nALL DONE. Pull this folder back to your laptop:')
print('  ' + DRIVE_RUN_DIR)
print('\nContents:')
subprocess.run(['ls', '-laR', DRIVE_RUN_DIR], check=False)



In [ ]:
# --- 18. Print final results summary (text table) ---
import json

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

ret = load_jsonl('outputs/retrieval_results.jsonl')
rer = load_jsonl('outputs/reranked_results.jsonl')

print(f"retrieval: {len(ret)} queries | reranked: {len(rer)} queries")

by_type = {}
for r in ret:
    by_type.setdefault(r['query_type'], []).append(r)
for qt, rows in by_type.items():
    sizes = [len(r['results']) for r in rows]
    print(f"{qt}: n={len(rows)}, pool size range=({min(sizes)},{max(sizes)})")

rer_map = {r['query_id']: r for r in rer}
changed = 0
for r in ret:
    rr = rer_map.get(r['query_id'])
    if not r['results'] or not rr or not rr['results']:
        continue
    if r['results'][0]['product_id'] != rr['results'][0]['product_id']:
        changed += 1
print(f"top-1 changed after reranking for {changed}/{len(ret)} queries")

print("\nnegation-constraint queries and their top-1 after rerank:")
for r in rer:
    q = queries_by_id_map.get(r['query_id']) if 'queries_by_id_map' in globals() else None
    text = (q or {}).get('query_text') or ''
    if ' not ' in text.lower() and r['results']:
        print(f"  {r['query_id']}: '{text}' -> {r['results'][0]['product_id']}")


In [ ]:
# --- 19. Inline visualization: before vs after rerank ---
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import io
from IPython.display import display, Image as IPImage, Markdown

ret_map = {r['query_id']: r for r in out_records}
rer_map = {r['query_id']: r for r in rerank_records}
queries_by_id_map = {q['query_id']: q for q in queries}

sample_qids = []
for qt in ("text", "image", "image_text"):
    for qid, q in queries_by_id_map.items():
        if q["query_type"] == qt:
            sample_qids.append(qid)
            break

rows_data = []
for qid in sample_qids:
    r2 = ret_map.get(qid)
    r3 = rer_map.get(qid)
    if not r2 or not r3 or not r2["results"] or not r3["results"]:
        continue
    top_before = r2["results"][:5]
    top_after = r3["results"][:5]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].bar(range(len(top_before)), [h["prefusion_score"] for h in top_before])
    axes[0].set_title(f"{qid} Stage2 prefusion")
    axes[1].bar(range(len(top_after)), [h["final_score"] for h in top_after])
    axes[1].set_title(f"{qid} Stage3 final")
    plt.tight_layout()
    plt.show()
    rows_data.append((qid, top_before[0]["product_id"], top_after[0]["product_id"]))

table = "\n".join(f"| {q} | {b} | {a} |" for q, b, a in rows_data)
display(Markdown("### Top-1 change table\n\n| query | before | after |\n|---|---|---|\n" + table))


In [ ]:
# --- 20. SELF-TEST: validate the whole pipeline output ---
import json, os, sys
from IPython.display import display, Markdown, HTML

def section(title):
    display(Markdown(f"## {title}"))

results_log = []

def check(name, cond, detail=''):
    results_log.append((name, bool(cond), detail))
    print(f"  [{'PASS' if cond else 'FAIL'}] {name}" + (f" ({detail})" if detail else ""))

section("Stage 2 + Stage 3 self-test")

ret_all = [json.loads(l) for l in open("outputs/retrieval_results.jsonl", encoding="utf-8") if l.strip()]
rer_all = [json.loads(l) for l in open("outputs/reranked_results.jsonl", encoding="utf-8") if l.strip()]
queries_all = [json.loads(l) for l in open("data/queries.jsonl", encoding="utf-8") if l.strip()]
catalog_chk = pd.read_parquet("data/catalog_subset.parquet")
catalog_ids_chk = set(catalog_chk["product_id"].tolist())
qmap = {q["query_id"]: q for q in queries_all}
ret_map_chk = {r["query_id"]: r for r in ret_all}

check("retrieval covers every query", {r["query_id"] for r in ret_all} == set(qmap))
check("reranked covers every query", {r["query_id"] for r in rer_all} == set(qmap))

dup_ok = rank_ok = subset_ok = dense_range_ok = ce_var_ok = True
for r in rer_all:
    ids = [h["product_id"] for h in r["results"]]
    dup_ok = dup_ok and len(ids) == len(set(ids))
    ranks = [h["rank"] for h in r["results"]]
    rank_ok = rank_ok and ranks == list(range(1, len(ranks) + 1))
    ret_ids = {h["product_id"] for h in ret_map_chk[r["query_id"]]["results"]}
    subset_ok = subset_ok and set(ids).issubset(ret_ids)
    for h in r["results"]:
        d = h.get("dense_score")
        if d is not None:
            dense_range_ok = dense_range_ok and -1.0001 <= d <= 1.0001
    ces = [h.get("cross_encoder_score") for h in r["results"] if h.get("cross_encoder_score") is not None]
    if len(ces) > 1:
        ce_var_ok = ce_var_ok and (max(ces) - min(ces)) > 1e-6

check("no duplicate product_id per query", dup_ok)
check("ranks are consecutive starting at 1", rank_ok)
check("reranked ids are a subset of retrieval candidates", subset_ok)
check("dense scores in valid cosine range", dense_range_ok)
check("cross-encoder scores vary within a query", ce_var_ok)

neg_queries = [q for q in queries_all if " not " in (q.get("query_text") or "").lower()]
check("at least one negative-constraint query is present", len(neg_queries) >= 1, f"count={len(neg_queries)}")

n_pass = sum(1 for _, ok, _ in results_log if ok)
n_fail = len(results_log) - n_pass
display(Markdown(f"**{n_pass} passed, {n_fail} failed**"))


# Stage 4


# Stage 4: Evaluation

This section builds the annotation pool, checks the finished qrels, runs the metrics, and displays the reports. Run the evaluation cells after `data/qrels_pool.csv` has been reviewed and finalized into `data/qrels.jsonl`.

Stage 4 does not need a GPU or an LLM API.


In [ ]:
import collections, json
from pathlib import Path

queries = [json.loads(line) for line in open('data/queries.jsonl') if line.strip()]
qrels_path = Path('data/qrels.jsonl')
if not qrels_path.exists():
    print('data/qrels.jsonl is not present yet. Build and annotate data/qrels_pool.csv first.')
else:
    qrels = [json.loads(line) for line in qrels_path.open() if line.strip()]
    counts = collections.Counter(row['query_id'] for row in qrels)
    missing = sorted({q['query_id'] for q in queries} - set(counts))
    print(f'queries: {len(queries)}')
    print(f'qrels: {len(qrels)}')
    print(f'covered queries: {len(counts)}/{len(queries)}')
    print(f'missing queries: {missing}')
    if counts:
        print(f'judgments/query range: {min(counts.values())}..{max(counts.values())}')


## Human judgment pool

The next cell creates 20 pooled candidates per query from dense, sparse,
hybrid, cross-encoder, and fused rankings. Review each query/product pair, including the product text and image, then fill every `relevance` and `reason` field in the CSV.


In [ ]:
!python src/build_judgment_pool.py --pool_out data/qrels_pool.csv


In [ ]:
# Run after data/qrels_pool.csv has been annotated.
# !python src/build_judgment_pool.py \
#   --finalize data/qrels_pool.csv \
#   --qrels_out data/qrels.jsonl


## Evaluation after annotation

Run these cells after `data/qrels.jsonl` exists. The metrics and reports are only meaningful when the qrels have been reviewed.


In [ ]:
!python src/evaluate.py \
  --runs outputs/retrieval_results.jsonl outputs/reranked_results.jsonl


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('reports/evaluation_report.md').read()))


In [ ]:
display(Markdown(open('reports/error_analysis.md').read()))


In [ ]:
!python src/tests.py --stage 4


# Stage 5


# Stage 5: Local LLM and Optional QLoRA

Run this section in a Google Colab GPU runtime. It runs:

1. Qwen2.5-1.5B baseline,
2. Qwen2.5-3B baseline,
3. Qwen2.5-1.5B with an optional QLoRA adapter.

No hosted-LLM API key is required. Download the executed notebook and return bundle after the required baseline cells finish.


In [ ]:
# Upload and extract the Stage 5 Colab package.
from google.colab import files
import os

uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
!unzip -q -o "$zip_name" -d /content

for candidate in ['/content/project', '/content/projectf', '/content']:
    if os.path.isfile(os.path.join(candidate, 'pyproject.toml')) and os.path.isdir(os.path.join(candidate, 'src')):
        PROJECT_DIR = candidate
        break
else:
    raise FileNotFoundError('Could not find pyproject.toml and src/ after extraction.')

os.chdir(PROJECT_DIR)
print('project:', PROJECT_DIR)


In [ ]:
# Confirm the GPU and install the Stage 5 runtime dependencies.
!nvidia-smi

import subprocess
import sys

try:
    import tomllib
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tomli"])
    import tomli as tomllib

with open("pyproject.toml", "rb") as f:
    project_cfg = tomllib.load(f)

deps = list(project_cfg["project"].get("optional-dependencies", {}).get("stage5-llm", []))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *deps])
print("Stage 5 dependencies installed from pyproject.toml")


## A. Student baseline: Qwen2.5-1.5B

This establishes whether prompting alone is sufficient. Fallback outputs are
schema-correct, but they do not count as successful raw model JSON.


In [ ]:
# TODO: fill the baseline local-LLM model and generation settings before running.
LLM_MODEL_BASELINE = "..."
LLM_TOP_K = ...
LLM_MAX_NEW_TOKENS = ...
LLM_REPAIR_ATTEMPTS = ...

!python src/generate_answer.py \
  --llm_backend local \
  --llm_model "$LLM_MODEL_BASELINE" \
  --load_in_4bit \
  --top_k "$LLM_TOP_K" \
  --max_new_tokens "$LLM_MAX_NEW_TOKENS" \
  --local_json_repair_attempts "$LLM_REPAIR_ATTEMPTS" \
  --debug_raw_out reports/raw_llm_debug.local_qwen15b.jsonl \
  --out outputs/final_answers.local_qwen15b.jsonl \
  --summary reports/llm_output_summary.local_qwen15b.md

!python src/tests.py --stage 5 \
  --answers outputs/final_answers.local_qwen15b.jsonl \
  --llm_summary reports/llm_output_summary.local_qwen15b.md \
  | tee reports/tests.stage5.local_qwen15b.txt


## B. Stronger reference: Qwen2.5-3B

The 3B model remains feasible on a T4 with 4-bit loading and generally follows
the schema more reliably.


In [ ]:
# TODO: fill the stronger local-LLM model and generation settings before running.
LLM_MODEL_STRONGER = "..."
LLM_TOP_K = ...
LLM_MAX_NEW_TOKENS = ...
LLM_REPAIR_ATTEMPTS = ...

!python src/generate_answer.py \
  --llm_backend local \
  --llm_model "$LLM_MODEL_STRONGER" \
  --load_in_4bit \
  --top_k "$LLM_TOP_K" \
  --max_new_tokens "$LLM_MAX_NEW_TOKENS" \
  --local_json_repair_attempts "$LLM_REPAIR_ATTEMPTS" \
  --debug_raw_out reports/raw_llm_debug.local_qwen3b.jsonl \
  --out outputs/final_answers.local_qwen3b.jsonl \
  --summary reports/llm_output_summary.local_qwen3b.md

!python src/tests.py --stage 5 \
  --answers outputs/final_answers.local_qwen3b.jsonl \
  --llm_summary reports/llm_output_summary.local_qwen3b.md \
  | tee reports/tests.stage5.local_qwen3b.txt


## Required baseline result

The next executed cell validates both local-model runs, saves their comparison, and packages the returned baseline artifacts. The optional QLoRA section starts afterward and has not been run.


In [ ]:
import os
os.chdir(PROJECT_DIR if 'PROJECT_DIR' in globals() else os.getcwd())
print('project:', os.getcwd())

!python src/tests.py --stage 5 \
  --answers outputs/final_answers.local_qwen15b.jsonl \
  --llm_summary reports/llm_output_summary.local_qwen15b.md \
  | tee reports/tests.stage5.local_qwen15b.txt

!python src/tests.py --stage 5 \
  --answers outputs/final_answers.local_qwen3b.jsonl \
  --llm_summary reports/llm_output_summary.local_qwen3b.md \
  | tee reports/tests.stage5.local_qwen3b.txt

import json
import pandas as pd

def summarize_debug_log(path, label):
    if not os.path.exists(path):
        return {"model": label, "path": path, "total": 0, "direct_valid": 0,
                "canonicalized": 0, "accepted": 0, "fallback": 0}
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    total = len(rows)
    direct_valid = sum(1 for r in rows if r.get("json_valid_first_try"))
    canonicalized = sum(1 for r in rows if r.get("repaired") and not r.get("used_fallback"))
    fallback = sum(1 for r in rows if r.get("used_fallback"))
    accepted = total - fallback
    return {"model": label, "path": path, "total": total, "direct_valid": direct_valid,
            "canonicalized": canonicalized, "accepted": accepted, "fallback": fallback}

rows = [
    summarize_debug_log("reports/raw_llm_debug.local_qwen15b.jsonl", "Qwen2.5-1.5B"),
    summarize_debug_log("reports/raw_llm_debug.local_qwen3b.jsonl", "Qwen2.5-3B"),
]
df_cmp = pd.DataFrame(rows)
df_cmp.to_csv("reports/stage5_model_comparison.csv", index=False)
print(df_cmp.to_string(index=False))

!zip -q -r /content/stage5_baseline_return_bundle.zip \
  outputs/final_answers.local_qwen15b.jsonl \
  outputs/final_answers.local_qwen3b.jsonl \
  reports/llm_output_summary.local_qwen15b.md \
  reports/llm_output_summary.local_qwen3b.md \
  reports/raw_llm_debug.local_qwen15b.jsonl \
  reports/raw_llm_debug.local_qwen3b.jsonl \
  reports/tests.stage5.local_qwen15b.txt \
  reports/tests.stage5.local_qwen3b.txt \
  reports/stage5_model_comparison.csv
print("wrote /content/stage5_baseline_return_bundle.zip")


## C. Advanced option: 1.5B QLoRA

The 3B accepted outputs become formatting targets. Training loss is applied
only to the target answer tokens. With 20 examples this demonstrates the
mechanics; it is not a large enough dataset for broad quality claims.


In [ ]:
# TODO: fill the SFT data source, base model, adapter path, and LoRA training
# settings before running the optional section.
SFT_SOURCE = "..."
LORA_BASE_MODEL = "..."
LORA_OUT_DIR = "..."
LORA_EPOCHS = ...
LORA_LR = "..."

!python src/build_answer_sft_data.py \
  --answers "$SFT_SOURCE" \
  --out data/answer_sft_seed.local_qwen3b.jsonl \
  --target_source existing_or_fallback

!python src/train_answer_lora.py \
  --train_jsonl data/answer_sft_seed.local_qwen3b.jsonl \
  --base_model "$LORA_BASE_MODEL" \
  --out_dir "$LORA_OUT_DIR" \
  --num_train_epochs "$LORA_EPOCHS" \
  --learning_rate "$LORA_LR"


In [ ]:
# TODO: fill the adapter path and generation settings for the LoRA run.
LORA_BASE_MODEL = "..."
LORA_OUT_DIR = "..."
LLM_TOP_K = ...
LLM_MAX_NEW_TOKENS = ...
LLM_REPAIR_ATTEMPTS = ...

!python src/generate_answer.py \
  --llm_backend local \
  --llm_model "$LORA_BASE_MODEL" \
  --adapter_path "$LORA_OUT_DIR" \
  --load_in_4bit \
  --top_k "$LLM_TOP_K" \
  --max_new_tokens "$LLM_MAX_NEW_TOKENS" \
  --local_json_repair_attempts "$LLM_REPAIR_ATTEMPTS" \
  --debug_raw_out reports/raw_llm_debug.local_qwen15b_lora.jsonl \
  --out outputs/final_answers.local_qwen15b_lora.jsonl \
  --summary reports/llm_output_summary.local_qwen15b_lora.md

!python src/tests.py --stage 5 \
  --answers outputs/final_answers.local_qwen15b_lora.jsonl \
  --llm_summary reports/llm_output_summary.local_qwen15b_lora.md \
  | tee reports/tests.stage5.local_qwen15b_lora.txt


In [ ]:
# Compact comparison from raw debug logs.
# NOTE: this cell is part of the optional GRPO/SFT extra-credit stretch goal
# (running src/build_answer_sft_data.py + src/train_answer_lora.py in cells
# 77-78). Those two training scripts are not part of the required Stage 5
# deliverable and are left for students who want to attempt the extra credit;
# this cell only compares whatever debug logs already exist on disk.
import json, os, pandas as pd

def summarize_debug_log(path, label):
    if not os.path.exists(path):
        return {"model": label, "path": path, "total": 0, "direct_valid": 0,
                "canonicalized": 0, "accepted": 0, "fallback": 0}
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    total = len(rows)
    direct_valid = sum(1 for r in rows if r.get("json_valid_first_try"))
    canonicalized = sum(1 for r in rows if r.get("repaired") and not r.get("used_fallback"))
    fallback = sum(1 for r in rows if r.get("used_fallback"))
    accepted = total - fallback
    return {"model": label, "path": path, "total": total, "direct_valid": direct_valid,
            "canonicalized": canonicalized, "accepted": accepted, "fallback": fallback}

rows = [
    summarize_debug_log("reports/raw_llm_debug.local_qwen15b.jsonl", "Qwen2.5-1.5B (baseline)"),
    summarize_debug_log("reports/raw_llm_debug.local_qwen3b.jsonl", "Qwen2.5-3B (baseline)"),
    summarize_debug_log("reports/raw_llm_debug.local_qwen15b_lora.jsonl", "Qwen2.5-1.5B + LoRA"),
]
df_cmp = pd.DataFrame(rows)
df_cmp.to_csv("reports/stage5_model_comparison_with_lora.csv", index=False)
print(df_cmp.to_string(index=False))


## Optional LoRA return artifacts

Run this only after completing the optional LoRA section.


In [ ]:
# TODO: package the optional LoRA return artifacts after that section runs.
!zip -q -r /content/stage5_lora_return_bundle.zip \
  outputs/final_answers.local_qwen15b_lora.jsonl \
  reports/llm_output_summary.local_qwen15b_lora.md \
  reports/raw_llm_debug.local_qwen15b_lora.jsonl \
  reports/tests.stage5.local_qwen15b_lora.txt \
  reports/stage5_model_comparison_with_lora.csv \
  data/answer_sft_seed.local_qwen3b.jsonl \
  adapters/stage5-qwen25-15b-json-lora
files.download('/content/stage5_lora_return_bundle.zip')
